In [10]:
import opendatasets as od

od.download("https://www.kaggle.com/competitions/home-credit-default-risk/data", download_dir="./data/inputs")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username:Your Kaggle Key:Downloading home-credit-default-risk.zip to .\home-credit-default-risk


100%|██████████| 688M/688M [00:24<00:00, 29.7MB/s] 



Extracting archive .\home-credit-default-risk/home-credit-default-risk.zip to .\home-credit-default-risk


In [1]:
import pandas as pd 


### Merging Data

In [2]:
bureau_df = pd.read_csv("../data/inputs/home-credit-default-risk/bureau.csv")
raw_train_df = pd.read_csv("../data/inputs/home-credit-default-risk/application_train.csv").dropna(subset=["TARGET"])
raw_test_df = pd.read_csv("../data/inputs/home-credit-default-risk/application_test.csv")

len(bureau_df), len(raw_train_df), len(raw_test_df)

(1716428, 307511, 48744)

In [3]:
raw_train_df['TARGET'].value_counts()

TARGET
0    282686
1     24825
Name: count, dtype: int64

In [4]:
bureau_agg = bureau_df.groupby("SK_ID_CURR").agg({
    "SK_ID_BUREAU": "count",                          # number of credits
    "AMT_CREDIT_SUM": ["mean", "sum"],                # exposure
    "AMT_CREDIT_SUM_DEBT": "sum",                     # total debt
    "AMT_CREDIT_SUM_OVERDUE": "sum",                  # total overdue
    "CREDIT_DAY_OVERDUE": "max",                      # worst delay
    "DAYS_CREDIT": "mean"                             # recency
}).rename(columns={
    "SK_ID_BUREAU": "bureau_count",
    "AMT_CREDIT_SUM": "avg_credit_sum",
    "AMT_CREDIT_SUM_DEBT": "total_debt",
    "AMT_CREDIT_SUM_OVERDUE": "total_overdue",
    "CREDIT_DAY_OVERDUE": "max_overdue",
    "DAYS_CREDIT": "avg_credit_days"
})

In [6]:
bureau_agg.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col 
                      for col in bureau_agg.columns]

# Reset the index if SK_ID_CURR is currently the index
if 'SK_ID_CURR' not in bureau_agg.columns:
    bureau_agg = bureau_agg.reset_index()

# Now the merge will work perfectly
train = raw_train_df.merge(bureau_agg, on="SK_ID_CURR", how="left")
test  = raw_test_df.merge(bureau_agg, on="SK_ID_CURR", how="left")

In [7]:
bureau_df.columns

Index(['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY',
       'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE',
       'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG',
       'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT',
       'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE',
       'AMT_ANNUITY'],
      dtype='str')

In [8]:
raw_train_df.columns.to_list()

['SK_ID_CURR',
 'TARGET',
 'NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_A

In [11]:
train.isna().sum()[train.isna().sum() > 1000]

NAME_TYPE_SUITE           1292
OWN_CAR_AGE             202929
OCCUPATION_TYPE          96391
EXT_SOURCE_1            173378
EXT_SOURCE_3             60965
                         ...  
avg_credit_sum_sum       44020
total_debt_sum           44020
total_overdue_sum        44020
max_overdue_max          44020
avg_credit_days_mean     44020
Length: 69, dtype: int64